# Notebook 01-CIC — CIC-IDS2017 Data Exploration and Preprocessing

**Project:** Calibrated and Stability-Aware Explainable Intrusion Detection
**Author:** Md Anas Biswas, University of Portsmouth
**Dataset:** CIC-IDS2017 (2,830,743 flows across 5 days)

## What this notebook does

1. Loads all 8 daily CSV files from CIC-IDS2017
2. Cleans canonical CIC-IDS2017 issues: leading spaces in column names, inf/NaN values
3. **Stratified subsample to 200K** for tractable downstream computation
4. Maps 14+ attack labels to 5 categories aligned with NSL-KDD (Normal/DoS/Probe/R2L/U2R)
5. Stratified 80/20 train/test split (CIC-IDS2017 has no official split)
6. Standardises numerical features
7. Saves processed arrays to `data/processed/cic_ids2017/`

## Note on Engelen WTMC-2021 corrections

This notebook uses the **original UNB CIC-IDS2017** data (via the `shadman1028/cicids2017-official-flow-feature-csv-files` Kaggle mirror). The Engelen WTMC-2021 corrected version (which fixes label errors in the Bot day and duplicates) is **not applied here** due to time constraints. This is documented as a limitation for future work.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull
print(f'✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
Already up to date.
✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
import numpy as np
import pandas as pd
import json, pickle, glob, gc
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
np.random.seed(SEED)

RAW_DIR = Path(REPO) / 'data' / 'raw' / 'cic_ids2017'
OUT_DIR = Path(REPO) / 'data' / 'processed' / 'cic_ids2017'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = Path(REPO) / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Files in raw dir:')
for f in sorted(RAW_DIR.glob('*.csv')):
    size_mb = f.stat().st_size / 1024**2
    print(f'  {f.name}  ({size_mb:.1f} MB)')

Files in raw dir:
  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv  (73.6 MB)
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv  (73.3 MB)
  Friday-WorkingHours-Morning.pcap_ISCX.csv  (55.6 MB)
  Monday-WorkingHours.pcap_ISCX.csv  (168.7 MB)
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv  (79.3 MB)
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv  (49.6 MB)
  Tuesday-WorkingHours.pcap_ISCX.csv  (128.8 MB)
  Wednesday-workingHours.pcap_ISCX.csv  (214.7 MB)


---
## Step 1 — Load and concatenate all 8 daily CSVs

Memory note: full 2.8M rows × 78 features ≈ 1.7 GB in memory. Colab Pro has 25 GB RAM so this is fine. We'll subsample shortly.

In [ ]:
csv_files = sorted(RAW_DIR.glob('*.csv'))

dfs = []
for f in csv_files:
    df = pd.read_csv(f, low_memory=False)
    print(f'  {f.name:<60}: {df.shape}')
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
del dfs; gc.collect()
print(f'\nCombined shape: {df_all.shape}')
print(f'Memory: {df_all.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv            : (225745, 79)
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv        : (286467, 79)
  Friday-WorkingHours-Morning.pcap_ISCX.csv                   : (191033, 79)
  Monday-WorkingHours.pcap_ISCX.csv                           : (529918, 79)
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv : (288602, 79)
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv      : (170366, 79)


In [ ]:
# Fix column names — strip leading/trailing whitespace (CIC-IDS2017 has many ' Label' style columns)
df_all.columns = df_all.columns.str.strip()

# Show columns
print(f'Total columns: {len(df_all.columns)}')
print(f'\nFirst 10 columns: {list(df_all.columns[:10])}')
print(f'Last column (should be Label): {df_all.columns[-1]}')

---
## Step 2 — Clean inf/NaN values

CIC-IDS2017 has known data quality issues: flow rate features (packets/sec, bytes/sec) can be infinity when duration=0, and some rows have NaN.

In [ ]:
# Replace inf with NaN, then drop rows with NaN
n_before = len(df_all)
df_all = df_all.replace([np.inf, -np.inf], np.nan)
n_inf = n_before - df_all.dropna().shape[0]
df_all = df_all.dropna()
df_all = df_all.reset_index(drop=True)

print(f'Rows before cleaning: {n_before:,}')
print(f'Rows dropped (inf/NaN): {n_inf:,}')
print(f'Rows remaining: {len(df_all):,}')

---
## Step 3 — Inspect labels and map to 5 categories

Map CIC-IDS2017's 14+ attack types to the same 5-class taxonomy as NSL-KDD for cross-dataset comparability.

In [ ]:
print('Original label distribution:')
print(df_all['Label'].value_counts())

# Map CIC-IDS2017 labels to 5-class taxonomy (same as NSL-KDD)
LABEL_MAPPING = {
    # Normal
    'BENIGN': 'Normal',

    # DoS-like (denial of service)
    'DoS Hulk': 'DoS',
    'DoS GoldenEye': 'DoS',
    'DoS slowloris': 'DoS',
    'DoS Slowhttptest': 'DoS',
    'DDoS': 'DoS',
    'Heartbleed': 'DoS',  # OpenSSL vulnerability, often grouped here

    # Probe (scanning / reconnaissance)
    'PortScan': 'Probe',

    # R2L (remote-to-local) — unauthorised remote access
    'FTP-Patator': 'R2L',
    'SSH-Patator': 'R2L',
    'Web Attack \x96 Brute Force': 'R2L',
    'Web Attack \x96 XSS': 'R2L',
    'Web Attack \x96 Sql Injection': 'R2L',
    'Bot': 'R2L',

    # U2R (user-to-root) — privilege escalation
    'Infiltration': 'U2R',
}

df_all['Category'] = df_all['Label'].map(LABEL_MAPPING)

unmapped = df_all[df_all['Category'].isnull()]['Label'].unique()
if len(unmapped) > 0:
    print(f'\n⚠ Unmapped labels: {unmapped}')
    # Drop unmapped rows
    n_before = len(df_all)
    df_all = df_all.dropna(subset=['Category']).reset_index(drop=True)
    print(f'  Dropped {n_before - len(df_all):,} unmapped rows')

print(f'\n5-class distribution:')
print(df_all['Category'].value_counts())

---
## Step 4 — Stratified subsample to 200K

For tractable downstream computation. Preserve all rare-class samples; subsample BENIGN heavily.

In [ ]:
TARGET_TOTAL = 200_000
MIN_PER_CLASS = 1_000  # ensure even rare classes have enough

category_counts = df_all['Category'].value_counts().to_dict()
print(f'Original counts:')
for cat, n in category_counts.items():
    print(f'  {cat}: {n:,}')

# Proportional subsample with floor
total = sum(category_counts.values())
subsample_counts = {}
for cat, n in category_counts.items():
    proportional = int(n / total * TARGET_TOTAL)
    floor = MIN_PER_CLASS
    take = min(n, max(proportional, floor))
    subsample_counts[cat] = take

print(f'\nTarget subsample counts:')
for cat, n in subsample_counts.items():
    print(f'  {cat}: {n:,}')

# Sample
rng = np.random.RandomState(SEED)
subsamples = []
for cat, n_take in subsample_counts.items():
    pool = df_all[df_all['Category'] == cat]
    if len(pool) <= n_take:
        subsamples.append(pool)
    else:
        subsamples.append(pool.sample(n=n_take, random_state=SEED))

df_sub = pd.concat(subsamples, ignore_index=True)
df_sub = df_sub.sample(frac=1, random_state=SEED).reset_index(drop=True)  # shuffle

del df_all; gc.collect()

print(f'\nFinal subsample: {len(df_sub):,} rows')
print(df_sub['Category'].value_counts())

---
## Step 5 — Encode features and labels

In [ ]:
# Separate features and labels
feature_cols = [c for c in df_sub.columns if c not in ['Label', 'Category']]
X = df_sub[feature_cols].values.astype(np.float32)

print(f'Feature matrix shape: {X.shape}')
print(f'Feature columns: {feature_cols[:5]}... (+{len(feature_cols)-5} more)')

# Binary label
y_binary = (df_sub['Category'] != 'Normal').astype(np.int64).values

# 5-class label
CATEGORY_TO_INT = {'Normal': 0, 'DoS': 1, 'Probe': 2, 'R2L': 3, 'U2R': 4}
INT_TO_CATEGORY = {v: k for k, v in CATEGORY_TO_INT.items()}
y_5class = df_sub['Category'].map(CATEGORY_TO_INT).astype(np.int64).values

print(f'\nBinary distribution: {np.bincount(y_binary)}')
print(f'  → Normal={np.sum(y_binary==0):,}, Attack={np.sum(y_binary==1):,}')
print(f'\n5-class distribution:')
for i in range(5):
    print(f'  {i} = {INT_TO_CATEGORY[i]:8s}  → {np.sum(y_5class==i):>7,}')

---
## Step 6 — Stratified train/test split + standardise

In [ ]:
# Stratified by 5-class so all classes appear in both splits
idx_all = np.arange(len(X))
idx_train, idx_test = train_test_split(
    idx_all, test_size=0.20, stratify=y_5class, random_state=SEED
)

X_train_raw = X[idx_train]
X_test_raw  = X[idx_test]
y_train_binary = y_binary[idx_train]
y_test_binary  = y_binary[idx_test]
y_train_5class = y_5class[idx_train]
y_test_5class  = y_5class[idx_test]

# Standardise (fit on train only)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_test  = scaler.transform(X_test_raw).astype(np.float32)

print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Train mean: {X_train.mean():.4f}, std: {X_train.std():.4f}')
print(f'Test mean: {X_test.mean():.4f}, std: {X_test.std():.4f}  (close to 0/1 but not exact — by design)')

---
## Step 7 — Visualise + save

In [ ]:
# Class distribution figure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

binary_train = pd.Series(y_train_binary).value_counts().sort_index()
binary_test = pd.Series(y_test_binary).value_counts().sort_index()
x = np.arange(2)
axes[0].bar(x - 0.2, binary_train.values, width=0.4, label='Train', color='#4C72B0')
axes[0].bar(x + 0.2, binary_test.values, width=0.4, label='Test', color='#DD8452')
axes[0].set_xticks(x); axes[0].set_xticklabels(['Normal', 'Attack'])
axes[0].set_title('CIC-IDS2017 — Binary class distribution')
axes[0].set_ylabel('Number of samples')
axes[0].legend()

names = [INT_TO_CATEGORY[i] for i in range(5)]
five_train = pd.Series(y_train_5class).value_counts().sort_index()
five_test = pd.Series(y_test_5class).value_counts().sort_index()
x = np.arange(5)
axes[1].bar(x - 0.2, five_train.values, width=0.4, label='Train', color='#4C72B0')
axes[1].bar(x + 0.2, five_test.values, width=0.4, label='Test', color='#DD8452')
axes[1].set_xticks(x); axes[1].set_xticklabels(names)
axes[1].set_title('CIC-IDS2017 — 5-class distribution (log scale)')
axes[1].set_ylabel('Samples (log)')
axes[1].set_yscale('log')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'cic_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save
np.save(OUT_DIR / 'X_train.npy', X_train)
np.save(OUT_DIR / 'X_test.npy', X_test)
np.save(OUT_DIR / 'y_train_binary.npy', y_train_binary)
np.save(OUT_DIR / 'y_test_binary.npy', y_test_binary)
np.save(OUT_DIR / 'y_train_5class.npy', y_train_5class)
np.save(OUT_DIR / 'y_test_5class.npy', y_test_5class)

with open(OUT_DIR / 'feature_names.json', 'w') as f:
    json.dump(feature_cols, f, indent=2)

class_info = {
    'binary': {'0': 'Normal', '1': 'Attack'},
    'multiclass_5': INT_TO_CATEGORY,
    'label_mapping': LABEL_MAPPING,
    'note': 'Engelen WTMC-2021 corrections not applied; future work.',
}
with open(OUT_DIR / 'class_mappings.json', 'w') as f:
    json.dump(class_info, f, indent=2)

with open(OUT_DIR / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print(f'✓ Saved to {OUT_DIR}\n')
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file():
        print(f'  {p.name:<30}  {p.stat().st_size/1024:>10,.1f} KB')

---
## Step 8 — Commit

In [ ]:
os.chdir(REPO)
!git add notebooks/01_cic_data_exploration.ipynb
!git add results/figures/cic_class_distribution.png
!git add data/processed/cic_ids2017/feature_names.json
!git add data/processed/cic_ids2017/class_mappings.json
!git status --short
!git commit -m 'Notebook 01-CIC: CIC-IDS2017 preprocessing (200K stratified subsample)'
!git push

---
## Summary

**Done:**
- ✓ Loaded all 8 daily CSVs (2.83M flows total)
- ✓ Cleaned inf/NaN (typically ~2k rows dropped)
- ✓ Mapped 14 attack labels to 5 categories aligned with NSL-KDD
- ✓ Stratified 200K subsample with rare-class floors
- ✓ 80/20 train/test split, stratified
- ✓ Standardised features
- ✓ Saved processed arrays

**Known limitations:**
- Engelen WTMC-2021 corrections not applied (download/processing time constraint)
- Subsampled to 200K rather than using full 2.83M (compute constraint)

**Next:** Reuse the same training/calibration/SHAP notebooks (02-04) on the new processed arrays.
